In [1]:
#import seaborn as sns
import pandas as pd
from pylab import *
%matplotlib inline
#from scipy.stats import gaussian_kde
from matplotlib.colors import ListedColormap
#from scipy.spatial import ConvexHull
#import statistics
#from scipy.spatial.distance import mahalanobis
from preprocessing_aux import *

In [2]:
infile = 'dataset_chm_shp.csv'
outfile = 'dataset_chm_shp_transformed.csv'
df = pd.read_csv(infile)

In [3]:
df['rc'] = df['R'] / (df['R'] + df['G'] + df['B'])
df['gc'] = df['G'] / (df['R'] + df['G'] + df['B'])

In [4]:
# --------------------------------------------------------------------------------------
# extending pada data frame by rc/gc, rc+gc, luminance Y, perceived lightness L, z_score_Y, z_score_L 
# Excess indecies ExG, ExR, ExB, indices ExGmExR, Ikaw, MGRVI, and GLI 
# --------------------------------------------------------------------------------------
# blue chromaticity
df['bc'] = 1-(df['rc']+df['gc'])

# ration rc/gc
df['rc/gc'] = df['rc']/df['gc']

# sum rc+gc
df['rc+gc'] = df['rc']+df['gc']

# excess indecies
ExG = [excess_index(G=val, R=np.array(df['R'])[i], B=np.array(df['B'])[i], convert_to_255=False)
       for i, val in enumerate(np.array(df['G']))]
df['ExG'] = ExG

ExR = [excess_index(G=val, R=np.array(df['R'])[i], B=np.array(df['B'])[i], convert_to_255=False, excessband='red')
       for i, val in enumerate(np.array(df['G']))]
df['ExR'] = ExR

ExB = [excess_index(G=val, R=np.array(df['R'])[i], B=np.array(df['B'])[i], convert_to_255=False, excessband='blue')
       for i, val in enumerate(np.array(df['G']))]
df['ExB'] = ExB

df.head()

# index ExGmExR
df['ExGmExR'] = df['ExG']-df['ExR']

# index Ikaw - Kawashima Index
df['Ikaw'] = (df['R']-df['B'])/(df['R']+df['B'])

# MGRVI - modified green red vegetation index
df['MGRVI'] = ((df['G'])**2 - (df['R'])**2)/((df['G'])**2 + (df['R'])**2)

# GLI - green leaf index
df['GLI'] = (2*df['G'] - df['R'] - df['B'])/(2*df['G'] + df['R'] + df['B'])

# luminance Y
Y = [0.2126*sRGBtoLin(val) + 
     0.7152*sRGBtoLin(np.array(df['G'])[i]) + 
     0.0722*sRGBtoLin(np.array(df['B'])[i]) for i, val in enumerate(np.array(df['R']))]
df['Y'] = Y

# perceived lightness L
L = [YtoLstar(val) for val in df['Y']]
df['L'] = L

# z_score of the luminance Y
df['z_score_Y'] = 0
sites = set(df['site'])
for site in sites:
    condition = df['site'] == site
    subdata_site = df[df['site'].isin([site])]['Y']
    z_site = z_score(x_all=np.array(subdata_site))
    df.loc[condition, 'z_score_Y'] = z_site
    
# z_score of the perceived lightness L
df['z_score_L'] = 0
sites = set(df['site'])
for site in sites:
    condition = df['site'] == site
    subdata_site = df[df['site'].isin([site])]['L']
    z_site = z_score(x_all=np.array(subdata_site))
    df.loc[condition, 'z_score_L'] = z_site

print(df.head())

          R         G         B  chm  veg_class     site  chicken_0   
0  0.109804  0.239216  0.129412    1          5  CS-103A  -0.263335  \
1  0.082353  0.211765  0.105882    1          5  CS-103A  -0.263335   
2  0.098039  0.223529  0.113725    1          5  CS-103A  -0.263335   
3  0.082353  0.211765  0.101961    1          5  CS-103A  -0.263335   
4  0.388235  0.505882  0.364706    3          6  CS-103A   0.121613   

   chicken_1  chicken_2  chicken_3  ...       ExR       ExB   ExGmExR   
0  -0.924741   0.453007   0.793199  ... -0.085490 -0.058039  0.324706  \
1  -0.924741   0.453007   0.793199  ... -0.096471 -0.063529  0.331765   
2  -0.924741   0.453007   0.793199  ... -0.086275 -0.064314  0.321569   
3  -0.924741   0.453007   0.793199  ... -0.096471 -0.069020  0.335686   
4  -1.450172  -0.453014   0.432836  ...  0.037647  0.004706  0.221176   

       Ikaw     MGRVI       GLI         Y          L  z_score_Y  z_score_L  
0 -0.081967  0.651942  0.333333  0.036942  22.633457  -1.

In [5]:
df.to_csv(outfile, index=False)